# ARGO — zoom on the hands (D44), training on Colab

Trains the **nothing / pistol / knife / rifle** classifier on the crops around the wrists (see `armi/ingrandimento_mani` in `src/sicurezza/visione.py`).
The crops are ready in the private Kaggle dataset `marcozambrella/roma-safe-cctv-ritagli-mani`.

**Before starting:** *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all*.
Even before that: put your Kaggle API token in Colab's *Secrets* with the name `KAGGLE_API_TOKEN` (instructions below), or have a `kaggle.json` ready.
At the end the weights are uploaded to the private Kaggle dataset `marcozambrella/roma-safe-cctv-mani-modello` (and also downloaded to your computer).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q --depth 1 https://github.com/marcozambrella/argo /content/repo
!pip install -q "ultralytics>=8.4" kaggle
import sys; sys.path.insert(0, "/content/repo/src")

Kaggle credentials, **one of the two**:
- **new API token** (Kaggle → Settings → API → *Generate New Token*): 🔑 *Secrets* icon in the left bar → *Add new secret* → name `KAGGLE_API_TOKEN`, value = the token → enable *Notebook access*;
- or the old `kaggle.json`: if the secret is missing, the cell below asks you to upload it.

In [ ]:
import os, shutil
from google.colab import files, userdata
try:   # token nuovo di Kaggle, dai Secret di Colab (non resta scritto nel notebook)
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    print("uso il secret KAGGLE_API_TOKEN")
except Exception:   # secret assente o senza accesso: vecchio kaggle.json
    caricati = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    shutil.move(next(iter(caricati)), "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
!kaggle datasets list --mine | head -5

In [ ]:
!kaggle datasets download marcozambrella/roma-safe-cctv-ritagli-mani -p /content --unzip -q
!cd /content && (ls ritagli_mani.zip >/dev/null 2>&1 && unzip -q -o ritagli_mani.zip -d /content) ; ls /content/ritagli_mani
!for s in train val test; do for c in niente pistol knife rifle; do echo "$s $c $(ls /content/ritagli_mani/$s/$c 2>/dev/null | wc -l)"; done; done

Training (~30-60 minutes on a T4):

In [ ]:
!cd /content/repo && PYTHONPATH=src python -m sicurezza.visione addestra-mani --dati /content/ritagli_mani --uscita /content/runs --dispositivo 0 --epoche 30

Evaluation on the test crops:

In [ ]:
from ultralytics import YOLO
modello = YOLO("/content/runs/yolo26s_mani/weights/best.pt")
metriche = modello.val(data="/content/ritagli_mani", split="test", imgsz=224)
print("accuratezza top-1 sul test:", metriche.top1)

Upload the weights to Kaggle (private dataset) and download them here too:

In [ ]:
import json, shutil, subprocess
os.makedirs("/content/mani_modello", exist_ok=True)
shutil.copy("/content/runs/yolo26s_mani/weights/best.pt", "/content/mani_modello/mani_best.pt")
shutil.copy("/content/runs/yolo26s_mani/results.csv", "/content/mani_modello/results.csv")
json.dump({"title": "roma-safe-cctv mani modello", "id": "marcozambrella/roma-safe-cctv-mani-modello",
           "licenses": [{"name": "other"}]}, open("/content/mani_modello/dataset-metadata.json", "w"))
esiste = subprocess.run(["kaggle", "datasets", "status", "marcozambrella/roma-safe-cctv-mani-modello"],
                        capture_output=True, text=True).returncode == 0
comando = (["kaggle", "datasets", "version", "-p", "/content/mani_modello", "-m", "nuovo addestramento"] if esiste
           else ["kaggle", "datasets", "create", "-p", "/content/mani_modello"])
print(subprocess.run(comando, capture_output=True, text=True).stdout)
files.download("/content/mani_modello/mani_best.pt")